# PCU-JOINT-CROSS-LAYER-001 — Joint L15/L23 coordination

Engineering-only diagnostic. Reuse the exact published depth-3 L15/K16 and L23/K16 Cell IDs. Replay and freeze the exact L7/K64 hybrid state. Compare the published sequential L15->freeze->L23 control against two jointly optimized arms.

Primary: Joint-128 on GPU0, matching 128 updates per L15/L23 parameter. Secondary: Joint-256 on GPU1, extra joint adaptation diagnostic only. Formal seeds are never executed.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

BRANCH = 'codex/pcu-composability-kill-001'
REPO = Path('/kaggle/working/mini-cells')
OUT = REPO / 'artifacts/research/pcu-joint-cross-layer-001/engineering/26090501-l15k16-l23k16-joint'
DEPTH3 = REPO / 'artifacts/research/pcu-sparse-path-depth-001/engineering/26090501-depth3-4-5'
WORKERS = Path('/kaggle/working/pcu-joint-cross-layer-001-workers')
FORMAL_SEEDS = (26090511, 26090512, 26090513)
REQUIRED_TRANSFORMERS = '5.16.1'
os.environ.setdefault('HF_HOME', '/kaggle/working/hf-cache')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def run(cmd, *, env=None, capture=False):
    cmd = [str(x) for x in cmd]
    print('+', ' '.join(cmd))
    result = subprocess.run(cmd, check=True, env=env, text=True, capture_output=capture)
    return result.stdout.strip() if capture else ''

if not REPO.exists():
    run(['git', 'clone', '--branch', BRANCH, 'https://github.com/ArcheLabs/mini-cells.git', REPO])
os.chdir(REPO)
run(['git', 'fetch', 'origin'])
run(['git', 'checkout', BRANCH])
run(['git', 'pull', '--ff-only', 'origin', BRANCH])
run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'])
run([sys.executable, '-m', 'pip', 'install', f'transformers=={REQUIRED_TRANSFORMERS}', 'huggingface_hub>=0.36,<2.0', 'safetensors>=0.4', 'accelerate>=1.0'])

import torch, transformers
assert transformers.__version__ == REQUIRED_TRANSFORMERS
assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 2, f'Need 2 GPUs, found {torch.cuda.device_count()}'
print(json.dumps({
    'commit': run(['git', 'rev-parse', 'HEAD'], capture=True),
    'tree': run(['git', 'rev-parse', 'HEAD^{tree}'], capture=True),
    'gpu0': torch.cuda.get_device_name(0),
    'gpu1': torch.cuda.get_device_name(1),
    'transformers': transformers.__version__,
}, indent=2))


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
github_token = secrets.get_secret('GITHUB_TOKEN')
assert hf_token and github_token
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
os.environ['GITHUB_TOKEN'] = github_token
login(token=hf_token, add_to_git_credential=False)
print('Secrets loaded; token values were not printed.')


In [ ]:
SEED_REGISTRY = REPO / 'research/formal_seed_registry.json'
def formal_states():
    payload = json.loads(SEED_REGISTRY.read_text())
    return {int(row['seed']): row['state'] for row in payload['seeds']}
expected = {seed: 'RESERVED_UNTOUCHED' for seed in FORMAL_SEEDS}
assert formal_states() == expected
assert run(['git', 'hash-object', SEED_REGISTRY], capture=True) == '71a3015a7d54e795538b3aa6750860f0b9168cb3'
print(json.dumps({'formal_seed_states': formal_states()}, indent=2))


In [ ]:
required = ['DEPTH_3.json', 'DECISION.json']
assert not [name for name in required if not (DEPTH3 / name).is_file()]
depth = json.loads((DEPTH3 / 'DEPTH_3.json').read_text())
decision = json.loads((DEPTH3 / 'DECISION.json').read_text())
assert decision['status'] == 'DEEPER_SPARSE_PATH_DID_NOT_IMPROVE'
assert depth['topology']['layers'] == [7, 15, 23]
assert depth['topology']['transport_k'] == [16]
assert depth['topology']['readout_k'] == 16
assert abs(depth['metrics']['direct_accuracy'] - 0.140625) < 1e-12
assert abs(depth['metrics']['ranking_eval_accuracy'] - 0.7890625) < 1e-12
assert len(depth['stages']) == 2
assert depth['stages'][0]['layer'] == 15 and depth['stages'][0]['selected_k'] == 16
assert depth['stages'][1]['layer'] == 23 and depth['stages'][1]['selected_k'] == 16
remote = 'artifacts/research/pcu-sparse-path-depth-001/engineering/26090501-depth3-4-5/DEPTH_3.json'
assert subprocess.run(['git', 'show', f'origin/{BRANCH}:{remote}'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
print(json.dumps({
    'sequential_direct': depth['metrics']['direct_accuracy'],
    'sequential_ranking': depth['metrics']['ranking_eval_accuracy'],
    'l15_cells': depth['stages'][0]['selected_cells'],
    'l23_cells': depth['stages'][1]['selected_cells'],
}, indent=2))


In [ ]:
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(REPO / 'src')
test_env['PYTEST_DISABLE_PLUGIN_AUTOLOAD'] = '1'
run([sys.executable, '-m', 'pytest', '-q', 'tests/research/05-pcu-kill-001'], env=test_env)
run([sys.executable, '-m', 'compileall', '-q', 'src/minicells/pcu_kill_001', 'scripts/research'])
print('PCU joint cross-layer test/compile gate: PASS')


In [ ]:
if OUT.exists():
    existing = sorted(p.name for p in OUT.glob('*.json'))
    assert not existing, f'Joint output already exists; inspect before rerun: {existing}'
if WORKERS.exists():
    stale = sorted(p.name for p in WORKERS.glob('*.json'))
    assert not stale, f'Stale external worker evidence exists; inspect/remove intentionally: {stale}'
run([
    sys.executable, 'scripts/research/run_pcu_joint_cross_layer_001.py',
    '--seed', '26090501',
    '--depth3', DEPTH3,
    '--out', OUT,
    '--worker-root', WORKERS,
])


In [ ]:
required = ['RUN_IDENTITY.json', 'DESIGN.json', 'RESULT.json', 'DECISION.json', 'JOINT_128.json', 'JOINT_256.json']
missing = [name for name in required if not (OUT / name).is_file()]
assert not missing, missing
decision = json.loads((OUT / 'DECISION.json').read_text())
result = json.loads((OUT / 'RESULT.json').read_text())
assert decision['valid_run'] is True
assert decision['formal_execution_not_started'] is True
assert decision['exact_published_depth3_cells_reused'] is True
assert decision['no_reallocation'] is True
assert decision['l7_frozen_before_joint_training'] is True
assert decision['dual_gpu_execution_required'] is True
assert decision['primary_coordination_steps'] == 128
assert decision['secondary_extra_joint_steps'] == 256
assert formal_states() == expected
print(json.dumps({
    'status': decision['status'],
    'sequential_direct': decision['sequential_direct_accuracy'],
    'sequential_ranking': decision['sequential_ranking_accuracy'],
    'joint128_direct': decision['joint128_direct_accuracy'],
    'joint128_ranking': decision['joint128_ranking_accuracy'],
    'joint128_first_token_top1': decision['joint128_first_token_top1_accuracy'],
    'joint128_later_token_top1': decision['joint128_later_token_top1_accuracy'],
    'joint256_direct': decision['joint256_direct_accuracy'],
    'joint256_ranking': decision['joint256_ranking_accuracy'],
    'summary': result['summary'],
    'formal_seed_states': formal_states(),
}, indent=2))


In [ ]:
run([sys.executable, 'scripts/research/publish_pcu_joint_cross_layer_001.py', '--branch', BRANCH])
assert formal_states() == expected
print(json.dumps({'published': True, 'formal_seed_states': formal_states()}, indent=2))
